# Cleaning, loading and preprocessing: lab notebook

**Statistical Modelling and Forecasting, Foundations of time series**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

### 1. Load a series with a datetime index (weekly CO2 from a bundled dataset)

In [ ]:
# 1. Load a series with a datetime index (weekly CO2 from a bundled dataset)
raw = sm.datasets.co2.load_pandas().data
print(raw.index[:3].tolist())
print("rows:", len(raw), "| missing:", int(raw["co2"].isna().sum()), "| frequency:", raw.index.freq)
s = raw["co2"].asfreq("W-SAT")                     # make the frequency explicit
print("after asfreq:", len(s), "rows, missing:", int(s.isna().sum()))

### 2. Fill missing values in three ways and compare with the neighbours

In [ ]:
# 2. Fill missing values in three ways and compare with the neighbours
gap = s[s.isna()].index[0]
loc = s.index.get_loc(gap)
print("neighbours of the first gap:", s.iloc[loc - 1:loc + 2].round(1).tolist())
for name, f in [("ffill", s.ffill()), ("linear", s.interpolate()), ("mean", s.fillna(s.mean()))]:
    print(f"{name:7s} value at the gap: {f.loc[gap]:.2f}")

### 3. Outliers: z-score rule and IQR rule on a clean series with injected spikes

In [ ]:
# 3. Outliers: z-score rule and IQR rule on a clean series with injected spikes
rng = np.random.default_rng(5)
x = pd.Series(50 + 0.2 * np.arange(120) + rng.normal(0, 1.5, 120))
x.iloc[[30, 75]] += [25, -22]
z = (x - x.mean()) / x.std()
q1, q3 = x.quantile([0.25, 0.75]); iqr = q3 - q1
print("flagged by |z| > 3:", x.index[np.abs(z) > 3].tolist())
print("flagged by IQR rule:", x.index[(x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)].tolist())
print("(the trend widens the overall spread, so global rules miss both spikes)")

### 4. Rolling median (Hampel-style) replaces only the spikes and keeps the trend

In [ ]:
# 4. Rolling median (Hampel-style) replaces only the spikes and keeps the trend
med = x.rolling(7, center=True, min_periods=1).median()
mad = (x - med).abs().rolling(7, center=True, min_periods=1).median() * 1.4826
bad = (x - med).abs() > 3 * mad.clip(lower=1.5)
cleaned = x.where(~bad, med)
print("replaced positions:", x.index[bad].tolist())
print("max abs error after cleaning vs true line:", round(float((cleaned - (50 + 0.2 * np.arange(120))).abs().max()), 2))

### 5. Resample to monthly means and split chronologically

In [ ]:
# 5. Resample to monthly means and split chronologically
monthly = s.interpolate().resample("MS").mean()
train, test = monthly.iloc[:-24], monthly.iloc[-24:]
print("monthly points:", len(monthly), "| train until", train.index[-1].date(), "| test from", test.index[0].date())

## Expected output

Running all cells above should print:

```
[Timestamp('1958-03-29 00:00:00'), Timestamp('1958-04-05 00:00:00'), Timestamp('1958-04-12 00:00:00')]
rows: 2284 | missing: 59 | frequency: <Week: weekday=5>
after asfreq: 2284 rows, missing: 59
neighbours of the first gap: [316.9, nan, 317.5]
ffill   value at the gap: 316.90
linear  value at the gap: 317.20
mean    value at the gap: 340.14
flagged by |z| > 3: []
flagged by IQR rule: []
(the trend widens the overall spread, so global rules miss both spikes)
replaced positions: [30, 75]
max abs error after cleaning vs true line: 3.65
monthly points: 526 | train until 1999-12-01 | test from 2000-01-01
```

## Practice exercises

Try these in the cells below.

1. Increase the missing share to 25% with long gaps. Which fill method degrades first, and why?
2. Add outliers and compare "Clip at 3 SD" with "Rolling median filter". Which one removes the spike without damaging the trend?
3. Write the pandas commands that load a monthly CSV, make the date the index and fill short gaps by interpolation.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
